In [1]:
import io
import os
import glob
import boto3
import duckdb
import pandas as pd
from PIL import Image

In [2]:
S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://rustfs:9000")
BUCKET = "lakehouse"
SEQUENCES_DIR = "/data/local/visdrone/sequences"
ANNOTATIONS_DIR = "/data/local/visdrone/annotations"
FRAGMENT_FRAMES = 30

CATEGORY_NAMES = {
    0: "ignored", 1: "pedestrian", 2: "people", 3: "bicycle",
    4: "car", 5: "van", 6: "truck", 7: "tricycle",
    8: "awning-tricycle", 9: "bus", 10: "motor", 11: "others"
}

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
print("S3 client ready:", S3_ENDPOINT)

S3 client ready: http://rustfs:9000


In [3]:
def parse_annotations(ann_path):
    rows = []
    with open(ann_path) as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 10:
                continue
            rows.append({
                "frame_id": int(parts[0]),
                "target_id": int(parts[1]),
                "x": int(parts[2]),
                "y": int(parts[3]),
                "w": int(parts[4]),
                "h": int(parts[5]),
                "score": int(parts[6]),
                "category_id": int(parts[7]),
                "category": CATEGORY_NAMES.get(int(parts[7]), "unknown"),
                "truncation": int(parts[8]),
                "occlusion": int(parts[9]),
            })
    return pd.DataFrame(rows)

In [4]:
fragment_rows = []
annotation_rows = []

seq_dirs = sorted(glob.glob(f"{SEQUENCES_DIR}/*"))
print(f"Found {len(seq_dirs)} sequences")

for seq_dir in seq_dirs:
    seq_name = os.path.basename(seq_dir)
    ann_path = f"{ANNOTATIONS_DIR}/{seq_name}.txt"

    if not os.path.exists(ann_path):
        print(f"No annotation file for {seq_name}, skipping")
        continue

    print(f"Processing {seq_name}...")
    ann_df = parse_annotations(ann_path)
    frame_files = sorted(glob.glob(f"{seq_dir}/*.jpg"))

    for frame_path in frame_files:
        frame_name = os.path.basename(frame_path)
        key = f"assets/visdrone/frames/{seq_name}/{frame_name}"
        with open(frame_path, "rb") as f:
            s3.upload_fileobj(f, BUCKET, key)

    clip_uri = f"s3://{BUCKET}/assets/visdrone/frames/{seq_name}/"
    all_frame_ids = sorted(ann_df["frame_id"].unique()) if not ann_df.empty else []
    max_frame = max(all_frame_ids) if all_frame_ids else len(frame_files)

    fragment_id = 0
    start_frame = 1
    while start_frame <= max_frame:
        end_frame = start_frame + FRAGMENT_FRAMES - 1
        frag_ann = ann_df[
            (ann_df["frame_id"] >= start_frame) & (ann_df["frame_id"] <= end_frame)
        ]

        fragment_rows.append({
            "clip_uri": clip_uri,
            "sequence": seq_name,
            "fragment_id": fragment_id,
            "start_frame": start_frame,
            "end_frame": end_frame,
            "n_objects": len(frag_ann),
            "classes": str(sorted(frag_ann["category"].unique().tolist())) if not frag_ann.empty else "[]",
        })

        for _, row in frag_ann.iterrows():
            annotation_rows.append({
                "clip_uri": clip_uri,
                "sequence": seq_name,
                "fragment_id": fragment_id,
                "frame_id": row["frame_id"],
                "target_id": row["target_id"],
                "x": row["x"],
                "y": row["y"],
                "w": row["w"],
                "h": row["h"],
                "category": row["category"],
                "truncation": row["truncation"],
                "occlusion": row["occlusion"],
            })

        start_frame = end_frame + 1
        fragment_id += 1

print("Done uploading and indexing.")

Found 7 sequences
Processing uav0000086_00000_v...
Processing uav0000117_02622_v...
Processing uav0000137_00458_v...
Processing uav0000182_00000_v...
Processing uav0000268_05773_v...
Processing uav0000305_00000_v...
Processing uav0000339_00001_v...
Done uploading and indexing.


In [5]:
fragments_df = pd.DataFrame(fragment_rows)
annotations_df = pd.DataFrame(annotation_rows)
print(f"Fragments: {len(fragments_df)}")
print(f"Annotation rows: {len(annotations_df)}")
fragments_df.head()

Fragments: 99
Annotation rows: 118127


,clip_uri,sequence,fragment_id,start_frame,end_frame,n_objects,classes
0,s3://lakehouse/assets/visdrone/frames/uav00000...,uav0000086_00000_v,0,1,30,1080,"['motor', 'pedestrian', 'people']"
1,s3://lakehouse/assets/visdrone/frames/uav00000...,uav0000086_00000_v,1,31,60,1093,"['motor', 'pedestrian', 'people']"
2,s3://lakehouse/assets/visdrone/frames/uav00000...,uav0000086_00000_v,2,61,90,1114,"['motor', 'pedestrian', 'people']"
3,s3://lakehouse/assets/visdrone/frames/uav00000...,uav0000086_00000_v,3,91,120,1203,"['motor', 'pedestrian', 'people']"
4,s3://lakehouse/assets/visdrone/frames/uav00000...,uav0000086_00000_v,4,121,150,1318,"['ignored', 'motor', 'pedestrian', 'people']"


In [11]:
con = duckdb.connect()
con.execute(open("/workspace/sql/00_attach.sql").read())
con.execute("CREATE TABLE raw.visdrone_fragments AS SELECT * FROM fragments_df")
con.execute("CREATE TABLE raw.visdrone_annotations AS SELECT * FROM annotations_df")
print("Tables created.")

CatalogException: Catalog Error: Table with name "visdrone_fragments" already exists!

In [12]:
print(con.sql("FROM ducklake_snapshots('lake')").df())
print(con.sql("SELECT COUNT(*) FROM raw.visdrone_fragments").df())
print(con.sql("SELECT COUNT(*) FROM raw.visdrone_annotations").df())
con.sql("SELECT clip_uri, fragment_id, start_frame, end_frame, n_objects, classes FROM raw.visdrone_fragments LIMIT 5").df()
con.close()

   snapshot_id                    snapshot_time  schema_version  \
0            0 2026-06-25 19:47:58.592626+00:00               0   
1            1 2026-06-25 19:47:58.638004+00:00               1   
2            2 2026-06-25 19:47:58.647187+00:00               2   
3            3 2026-06-25 19:47:58.669349+00:00               3   
4            4 2026-06-25 22:18:44.460503+00:00               4   
5            5 2026-06-25 23:13:50.427815+00:00               5   
6            6 2026-06-25 23:13:50.502705+00:00               6   

                                             changes author commit_message  \
0                      {'schemas_created': ['main']}   None           None   
1                       {'schemas_created': ['raw']}   None           None   
2                    {'schemas_created': ['silver']}   None           None   
3                      {'schemas_created': ['gold']}   None           None   
4  {'tables_created': ['raw.coco_annotations'], '...   None           Non